In [1]:
from pathlib import Path
from os import remove

PROJECT_ROOT = Path(".").resolve()
# print(PROJECT_ROOT)

SQUARES_PATH = PROJECT_ROOT / "dataset_squares"

MANUAL_PATH = SQUARES_PATH / "manually_annotated" / "_unlabeled"
AUTO_PATH = SQUARES_PATH / "auto_annotated" / "_unlabeled"

print(MANUAL_PATH)
print(AUTO_PATH)

if not MANUAL_PATH.exists():
    raise Exception("Manually annotated images path does not exist")
if not AUTO_PATH.exists():
    raise Exception("Auto annotated images path does not exist")


/Users/chamster/gitClones/ChessPiecesCV/dataset_squares/manually_annotated/_unlabeled
/Users/chamster/gitClones/ChessPiecesCV/dataset_squares/auto_annotated/_unlabeled


In [2]:
manual_filenames = []
auto_filenames = []

for filepath in MANUAL_PATH.iterdir():
    manual_filenames.append(filepath.name)
for filepath in AUTO_PATH.iterdir():
    auto_filenames.append(filepath.name)

In [ ]:
og_manual_len = len(manual_filenames)
og_auto_len = len(auto_filenames)

manuals_del = [a for a in manual_filenames if a not in auto_filenames]
autos_del   = [a for a in auto_filenames if a not in manual_filenames]

print(f"Manually annotated files count: {str(og_manual_len):<6} File count to delete: {str(len(manuals_del)):<5}")
print(f"Auto annotated files count:     {  str(og_auto_len):<6} File count to delete: {str(len(autos_del)):<5}")
print()

for del_file in manuals_del:
    del_path = MANUAL_PATH / del_file
    remove(del_path)

for del_file in autos_del:
    del_path = AUTO_PATH / del_file
    remove(del_path)

del(manual_filenames)
del(auto_filenames)
del(manuals_del)
del(autos_del)


Manually annotated files count: 32512  File count to delete: 4544
Auto annotated files count:     27968  File count to delete: 0


In [5]:
filenames_to_compare = []
for filepath in MANUAL_PATH.iterdir():
    filenames_to_compare.append(filepath.name)

print(f"There are {len(filenames_to_compare)} files left to compare.")

There are 27968 files left to compare.


In [17]:
import cv2
import matplotlib.pyplot as plt
from typing import Literal

def plot_image(image) -> None:
    plt.imshow(image)

def bgr2rgb(image) -> None:
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return rgb_image

def plot_bgr(image) -> None:
    rgb_image = bgr2rgb(image)
    plot_image(rgb_image)

def plot_rgb(image) -> None:
    plot_image(image)

def plot_gray(image) -> None:
    plt.imshow(image, cmap="gray")

def bgr2grayscale(image) -> None:
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray_image

def plot_images_grid(
    imgs:        list[cv2.typing.MatLike] | tuple[cv2.typing.MatLike],
    color_types: list[Literal["gray", "rgb", "bgr"]] | tuple[Literal["gray", "rgb", "bgr"]],
    titles:      list[str] | tuple[str],
    rows:        int,
    cols:        int,
    chart_title: str|None = None) -> None:

    plt.figure(figsize=(20, 20))
    if rows == 2 and cols == 3:
        plt.figure(figsize=(24, 12))
    elif rows == 2 and cols == 2:
        plt.figure(figsize=(12, 8))

    plt.tight_layout()

    # rows, cols = 2, 3
    # print(len(imgs))
    if (len(color_types) == 1 and
        len(imgs) > 1):
        color = color_types[0]
        color_types = [color for _ in imgs]

    if chart_title is not None:
        plt.title(chart_title)

    for i in range(len(imgs)):
        plt.subplot(rows, cols, i + 1)
        plt.title(titles[i])

        color_type = color_types[i]
        if color_type == "gray":
            plot_gray(imgs[i])
        elif color_type == "bgr":
            plot_bgr(imgs[i])
        elif color_type == "rgb":
            plot_rgb(imgs[i])

        plt.axis("off")

    plt.show()

In [26]:
from skimage.metrics import structural_similarity as ssim

successes = 0
fails = 0

for filename in filenames_to_compare:
    test_path_manual = MANUAL_PATH / filename
    test_path_auto   = AUTO_PATH / filename

    test_img_manual = cv2.imread(test_path_manual)
    test_img_auto   = cv2.imread(test_path_auto)

    test_img_manual_gray = bgr2grayscale(test_img_manual)
    test_img_auto_gray = bgr2grayscale(test_img_auto)

    score, _ = ssim(test_img_manual_gray, test_img_auto_gray, full=True)

    # print(f"Similarity: {score}")
    if score >= 0.60:
        successes += 1
    else:
        fails += 1

print(f"{successes} images are correctly cut")
print(f"{fails} images are incorrectly cut")


915 images are correctly cut
27053 images are incorrectly cut
